# BeliefBench hosted API walkthrough
This notebook uses the thin SDK. The benchmark engine remains on the hosted service. Never paste an OpenAI key into a notebook; set it in the environment before starting Jupyter.

In [ ]:
# Install the client-only wheel from the public release
%pip install https://github.com/mfrdixon/BeliefBench-Python/releases/download/v0.4.0/beliefbench-0.4.0-py3-none-any.whl pandas matplotlib

In [ ]:
import os
from pathlib import Path
from beliefbench import BeliefBench
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY before starting Jupyter'
assert os.getenv('BELIEFBENCH_SERVICE_TOKEN'), 'Set BELIEFBENCH_SERVICE_TOKEN before starting Jupyter'
BASE_URL = os.getenv('BELIEFBENCH_API_URL', 'https://beliefbench-api.onrender.com')

## Define a small world
The per-call token cap is 200. The hosted service also evaluates the worst-case total budget before accepting the job.

In [ ]:
config = {
    'seed': 7, 'scenarios': 3, 'repeats': 2, 'trajectory_length': 3,
    'model': 'gpt-5.4-mini', 'llm_mode': 'api', 'max_output_tokens': 200,
    'standard_rag_top_k': 2, 'partial_omit': 1, 'entropy_bins': [0.45, 0.75],
    'world': {'latent_node': 'Condition', 'nodes': {
        'Condition': {'outcomes': ['Absent','Present'], 'cpt': [0.8,0.2]},
        'Test': {'outcomes': ['Negative','Positive'], 'parents': ['Condition'],
                 'cpt': [[0.95,0.05],[0.10,0.90]],
                 'templates': ['The test was negative.','The test was positive.']}
    }},
    'decision': {'actions': ['Do_Nothing','Investigate'], 'payoffs': [[1,-3],[-0.2,1]]}
}

In [ ]:
with BeliefBench(BASE_URL) as client:
    print(client.health())
    archive = client.run(config, 'beliefbench-results.zip', client_request_id='notebook-demo')
archive

In [ ]:
import zipfile
result_dir = Path('beliefbench-results')
with zipfile.ZipFile(archive) as zf:
    zf.extractall(result_dir)
sorted(str(p) for p in result_dir.rglob('*') if p.is_file())

In [ ]:
import pandas as pd
from IPython.display import Image, display
summary = pd.read_csv(result_dir / 'summary.csv')
display(summary)
display(Image(filename=str(result_dir / 'plots' / 'posterior_error.png')))